# Model 6: XGBoost with Final CV (Stage 1: Find "Recipe")

This notebook implements **Stage 1** of our final strategy. We will find the best hyperparameters by using a robust, all-seasons, drift-protected Cross-Validation.

This notebook will:
1.  Define a new `create_feature_set_v4` function. This function is our new standard and solves our two problems:
    * It takes a 3-year `train_window` (e.g., 2020-2022) to **prevent data drift**.
    * It takes a full 12-month `val_window` (e.g., 2023) to **validate on all seasons**.
2.  Create our 2 robust CV folds (Val 2023, Val 2022). **This stage does not use 2024 data**, as it is saved for our final model.
3.  Run Optuna on XGBoost to find the best hyperparameters based on the *average* score of these 2 folds.

In [21]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import seaborn as sns
import itertools
import optuna

# --- 1. Load All Raw Data ---
print("--- 1. Loading All Raw Data ---")
RECEIVALS_FILE = "data/kernel/receivals.csv"
PURCHASE_ORDERS_FILE = "data/kernel/purchase_orders.csv"
MAPPING_FILE = "data/prediction_mapping.csv"
MATERIALS_FILE = "data/extended/materials.csv"
TRANSPORT_FILE = "data/extended/transportation.csv"

try:
    df_receivals = pd.read_csv(RECEIVALS_FILE)
    df_po = pd.read_csv(PURCHASE_ORDERS_FILE)
    df_mapping = pd.read_csv(MAPPING_FILE)
    df_materials = pd.read_csv(MATERIALS_FILE)
    df_transport = pd.read_csv(TRANSPORT_FILE)
    print("All data files loaded successfully.")
except FileNotFoundError as e:
    print(f"Error: {e}")
    raise

# --- 2. Perform All Cleaning Steps ---
print("\n--- 2. Cleaning Data ---")
df_receivals['date_arrival'] = pd.to_datetime(df_receivals['date_arrival'], utc=True, errors='coerce')
df_receivals_cleaned = df_receivals.dropna(
    subset=['rm_id', 'product_id', 'purchase_order_id', 'net_weight']
).copy()
df_receivals_cleaned = df_receivals_cleaned[df_receivals_cleaned['net_weight'] > 0].copy()

material_map = df_materials[['product_id', 'rm_id', 'raw_material_format_type']].drop_duplicates()
product_to_rm_map = material_map.drop_duplicates(subset=['product_id'], keep='first')

df_transport_cleaned = df_transport[
    ['rm_id', 'purchase_order_id', 'purchase_order_item_no', 'transporter_name', 'net_weight']
].dropna(subset=['transporter_name']).copy()
df_transport_cleaned.rename(columns={'net_weight': 'transport_net_weight'}, inplace=True)
df_transport_cleaned = df_transport_cleaned.drop_duplicates(
    subset=['rm_id', 'purchase_order_id', 'purchase_order_item_no'], 
    keep='last'
)

df_po['delivery_date'] = pd.to_datetime(df_po['delivery_date'], errors='coerce', utc=True)
df_po['created_date_time'] = pd.to_datetime(df_po['created_date_time'], errors='coerce', utc=True)
df_po['modified_date_time'] = pd.to_datetime(df_po['modified_date_time'], errors='coerce', utc=True)
df_po_cleaned = df_po.dropna(subset=['unit_id', 'unit', 'created_date_time']).copy()
df_po_cleaned = df_po_cleaned[df_po_cleaned['unit'] == 'KG'].copy()
df_po_cleaned = df_po_cleaned[df_po_cleaned['quantity'] > 0].copy()

df_po_cleaned = pd.merge(
    df_po_cleaned,
    product_to_rm_map,
    on='product_id',
    how='left' 
)
df_po_cleaned = df_po_cleaned.dropna(subset=['rm_id'])
print("All data cleaned.")

--- 1. Loading All Raw Data ---
All data files loaded successfully.

--- 2. Cleaning Data ---
All data cleaned.


In [22]:
# --- 3. Define our NEW v4 Feature Factory (Solves Drift + Seasonality) ---
print("\n--- 3. Defining create_feature_set_v4 ---")

def create_feature_set_v4(train_start_str, train_end_str, val_start_str, val_end_str):
    """
    Generates a complete (X, y) feature set.
    - Features are built *only* from the 'train' window (solves drift).
    - Targets are built *only* from the 'val' window (solves seasonality).
    """
    print(f"--- Generating v4 set: Train={train_start_str}to{train_end_str}, Val={val_start_str}to{val_end_str} ---")
    
    TRAIN_START = pd.to_datetime(train_start_str, utc=True)
    TRAIN_END = pd.to_datetime(train_end_str, utc=True)
    VAL_START = pd.to_datetime(val_start_str, utc=True)
    VAL_END = pd.to_datetime(val_end_str, utc=True)
    
    # --- Create Validation (Target) Set ---
    all_rm_ids = df_receivals_cleaned['rm_id'].unique()
    val_dates = pd.date_range(start=VAL_START, end=VAL_END, freq='D', tz='UTC')
    val_universe = list(itertools.product(all_rm_ids, val_dates))
    df = pd.DataFrame(val_universe, columns=['rm_id', 'forecast_end_date'])
    
    # --- Define Historical Data (FROM TRAIN WINDOW ONLY) ---
    hist_receivals = df_receivals_cleaned[
        (df_receivals_cleaned['date_arrival'] >= TRAIN_START) &
        (df_receivals_cleaned['date_arrival'] <= TRAIN_END)
    ].copy()
    
    hist_po_base = df_po_cleaned[
        (df_po_cleaned['created_date_time'] >= TRAIN_START) &
        (df_po_cleaned['created_date_time'] <= TRAIN_END)
    ].copy()
    
    hist_po = pd.merge(
        hist_po_base,
        df_transport_cleaned[['rm_id', 'purchase_order_id', 'purchase_order_item_no', 'transporter_name']],
        on=['rm_id', 'purchase_order_id', 'purchase_order_item_no'],
        how='left'
    )
    hist_po['transporter_name'] = hist_po['transporter_name'].fillna('Transporter_Unknown')

    # --- Build Target (y) from Validation Window ---
    daily_actuals = df_receivals_cleaned[
        (df_receivals_cleaned['date_arrival'] >= VAL_START) &
        (df_receivals_cleaned['date_arrival'] <= VAL_END)
    ]
    daily_actuals_grouped = daily_actuals.groupby(
        ['rm_id', daily_actuals['date_arrival'].dt.date]
    )['net_weight'].sum().reset_index(name='daily_net_weight')
    daily_actuals_grouped['forecast_end_date'] = pd.to_datetime(daily_actuals_grouped['date_arrival'], utc=True)
    df = pd.merge(
        df, daily_actuals_grouped[['rm_id', 'forecast_end_date', 'daily_net_weight']],
        on=['rm_id', 'forecast_end_date'], how='left'
    )
    df['daily_net_weight'] = df['daily_net_weight'].fillna(0)
    df = df.sort_values(by=['rm_id', 'forecast_end_date'])
    df['y_cumulative_weight'] = df.groupby('rm_id')['daily_net_weight'].cumsum()
    
    # --- Build Features (X) from Training Window ---
    
    # Aggregated PO Features
    hist_po['f_po_lead_time_days'] = (hist_po['delivery_date'] - hist_po['created_date_time']).dt.days
    po_agg = hist_po.groupby(['rm_id', 'delivery_date']).agg(
        daily_po_quantity=('quantity', 'sum'),
        avg_lead_time=('f_po_lead_time_days', 'mean')
    ).reset_index()
    po_agg.rename(columns={'delivery_date': 'po_delivery_date'}, inplace=True)
    
    rm_ids_to_merge = df['rm_id'].unique()
    merged_df = pd.merge(
        df[['rm_id', 'forecast_end_date']],
        po_agg[po_agg['rm_id'].isin(rm_ids_to_merge)],
        on='rm_id', how='left'
    )
    merged_df_filtered = merged_df[merged_df['forecast_end_date'] >= merged_df['po_delivery_date']].copy()
    cumulative_features = merged_df_filtered.groupby(['rm_id', 'forecast_end_date']).agg(
        f_cumulative_po_quantity=('daily_po_quantity', 'sum'),
        f_avg_lead_time=('avg_lead_time', 'mean')
    ).reset_index()
    df = pd.merge(df, cumulative_features, on=['rm_id', 'forecast_end_date'], how='left')
    df['f_cumulative_po_quantity'] = df['f_cumulative_po_quantity'].fillna(0)
    df['f_avg_lead_time'] = df['f_avg_lead_time'].fillna(0)
    
    # Time-Based (relative to the validation date)
    df['f_month'] = df['forecast_end_date'].dt.month
    df['f_day_of_week'] = df['forecast_end_date'].dt.dayofweek
    df['f_day_of_year'] = df['forecast_end_date'].dt.dayofyear
    df['f_is_month_end'] = df['forecast_end_date'].dt.is_month_end.astype(str)

    # Entity & Lag (built from training window)
    df_train_merged_eda = pd.merge(
        hist_receivals,
        hist_po[['purchase_order_id', 'purchase_order_item_no', 'delivery_date', 'rm_id']],
        on=['purchase_order_id', 'purchase_order_item_no', 'rm_id'], how='inner'
    )
    df_train_merged_eda['delivery_lag_days'] = (
        df_train_merged_eda['date_arrival'] - df_train_merged_eda['delivery_date']
    ).dt.days
    rm_id_lag_map = df_train_merged_eda.groupby('rm_id')['delivery_lag_days'].median().reset_index(name='f_median_lag_days')
    df = pd.merge(df, rm_id_lag_map, on='rm_id', how='left')
    df['f_median_lag_days'] = df['f_median_lag_days'].fillna(0)

    # f_receivals_Nd (built from end of training window)
    hist_windows = [30, 90, 180]
    for days in hist_windows:
        hist_start_date_window = TRAIN_END - pd.Timedelta(days=days)
        window_data = hist_receivals[
            (hist_receivals['date_arrival'] >= hist_start_date_window) &
            (hist_receivals['date_arrival'] <= TRAIN_END)
        ]
        feature_name = f'f_receivals_{days}d'
        feature_map = window_data.groupby('rm_id')['net_weight'].sum().reset_index(name=feature_name)
        df = pd.merge(df, feature_map, on='rm_id', how='left')
        df[feature_name] = df[feature_name].fillna(0)
        
    # V3 Static Features (built from training window)
    rm_id_static_features = hist_po.drop_duplicates(subset=['rm_id'], keep='last')[[
        'rm_id', 'status', 'raw_material_format_type', 'transporter_name'
    ]]
    df = pd.merge(df, rm_id_static_features, on='rm_id', how='left')
    df['status'] = df['status'].fillna('Unknown')
    df['raw_material_format_type'] = df['raw_material_format_type'].fillna('Unknown')
    df['transporter_name'] = df['transporter_name'].fillna('Unknown')

    # Filter to active rm_ids
    rm_id_max_y = df.groupby('rm_id')['y_cumulative_weight'].max()
    active_rm_ids = rm_id_max_y[rm_id_max_y > 0].index
    df_filtered = df[df['rm_id'].isin(active_rm_ids)]
    
    print(f"--- Block for {val_start_str} complete. Final Shape: {df_filtered.shape} ---")
    # Return the filtered set for calculations, and the *unfiltered* set for aligning
    return df_filtered, df

# --- 4. Define our Metric Functions (Corrected) ---
def quantile_error_0_2_raw(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_pred[y_pred < 0] = 0
    loss = np.mean(np.maximum(0.2 * (y_true - y_pred), 0.8 * (y_pred - y_true)))
    return loss

def xgb_native_quantile_error(y_pred, y_true_dmatrix):
    y_true_labels = y_true_dmatrix.get_label()
    score = quantile_error_0_2_raw(y_true_labels, y_pred)
    return 'q0.2_error', score

print("--- 4. All functions and data defined. Setup Complete. ---")


--- 3. Defining create_feature_set_v4 ---
--- 4. All functions and data defined. Setup Complete. ---


## Step 2: Build Final Cross-Validation Folds

Now we will build our 2 new CV folds, per our **Stage 1** plan:
* **Fold 1:** Train on 2020-2022 (3 years), **Validate on all of 2023.**
* **Fold 2:** Train on 2019-2021 (3 years), **Validate on all of 2022.**

In [23]:
print("--- Building 2 'Full-Year' Folds for Cross-Validation ---")

# --- Fold 1: Validate on 2023 ---
print("\nBuilding Fold 1 (Val 2023)...")
# Training set for Fold 1
df_train_1_filtered, df_train_1_full = create_feature_set_v4(
    '2020-01-01', '2022-12-31', # Train window
    '2020-01-01', '2022-12-31'  # Target window (matches train)
)
# Validation set for Fold 1
df_val_1_filtered, df_val_1_full = create_feature_set_v4(
    '2020-01-01', '2022-12-31', # Train window (features from past)
    '2023-01-01', '2023-12-31'  # Target window (predicting future)
)

# --- Fold 2: Validate on 2022 ---
print("\nBuilding Fold 2 (Val 2022)...")
# Training set for Fold 2
df_train_2_filtered, df_train_2_full = create_feature_set_v4(
    '2019-01-01', '2021-12-31', # Train window
    '2019-01-01', '2021-12-31'  # Target window (matches train)
)
# Validation set for Fold 2
df_val_2_filtered, df_val_2_full = create_feature_set_v4(
    '2019-01-01', '2021-12-31', # Train window (features from past)
    '2022-01-01', '2022-12-31'  # Target window (predicting future)
)

print("\n--- All 2 Folds Created. Aligning columns... ---")

# --- Align all folds ---
# We align using the FULL dataframes to capture all possible columns
all_dfs = [df_train_1_full, df_val_1_full, df_train_2_full, df_val_2_full]
CATEGORICAL_COLS = ['f_is_month_end', 'status', 'raw_material_format_type', 'transporter_name']

for i, df in enumerate(all_dfs):
    all_dfs[i] = pd.get_dummies(df, columns=CATEGORICAL_COLS, prefix_sep='_f_')

master_df = pd.concat(all_dfs, ignore_index=True).fillna(0)
ALL_FEATURES = [col for col in master_df.columns if col.startswith('f_')]
TARGET_COL = 'y_cumulative_weight'

# Store the final (X, y) pairs in a list
cv_folds = []

# Now, we use the FILTERED dataframes (which have active rm_ids)
# but align them to the MASTER list of columns
all_dfs_filtered = [
    pd.get_dummies(df_train_1_filtered, columns=CATEGORICAL_COLS, prefix_sep='_f_'),
    pd.get_dummies(df_val_1_filtered, columns=CATEGORICAL_COLS, prefix_sep='_f_'),
    pd.get_dummies(df_train_2_filtered, columns=CATEGORICAL_COLS, prefix_sep='_f_'),
    pd.get_dummies(df_val_2_filtered, columns=CATEGORICAL_COLS, prefix_sep='_f_')
]

def align_and_get_xy(df, master_cols, feature_cols, target_col):
    df_aligned, _ = df.align(master_cols, join='right', axis=1, fill_value=0)
    return df_aligned[feature_cols], df_aligned[target_col]

master_cols_df = pd.DataFrame(columns=master_df.columns)

# Create the final (X, y) tuples
X_t1, y_t1 = align_and_get_xy(all_dfs_filtered[0], master_cols_df, ALL_FEATURES, TARGET_COL)
X_v1, y_v1 = align_and_get_xy(all_dfs_filtered[1], master_cols_df, ALL_FEATURES, TARGET_COL)
X_t2, y_t2 = align_and_get_xy(all_dfs_filtered[2], master_cols_df, ALL_FEATURES, TARGET_COL)
X_v2, y_v2 = align_and_get_xy(all_dfs_filtered[3], master_cols_df, ALL_FEATURES, TARGET_COL)

cv_folds = [
    (X_t1, y_t1, X_v1, y_v1),
    (X_t2, y_t2, X_v2, y_v2)
]

print(f"\n--- Cross-Validation Folds Ready ---")
print(f"Total features (after one-hot): {len(ALL_FEATURES)}")
print(f"Fold 1 (Val 2023): Train={len(X_t1)}, Val={len(X_v1)}")
print(f"Fold 2 (Val 2022): Train={len(X_t2)}, Val={len(X_v2)}")

--- Building 2 'Full-Year' Folds for Cross-Validation ---

Building Fold 1 (Val 2023)...
--- Generating v4 set: Train=2020-01-01to2022-12-31, Val=2020-01-01to2022-12-31 ---
--- Block for 2020-01-01 complete. Final Shape: (63568, 17) ---
--- Generating v4 set: Train=2020-01-01to2022-12-31, Val=2023-01-01to2023-12-31 ---
--- Block for 2023-01-01 complete. Final Shape: (20075, 17) ---

Building Fold 2 (Val 2022)...
--- Generating v4 set: Train=2019-01-01to2021-12-31, Val=2019-01-01to2021-12-31 ---
--- Block for 2019-01-01 complete. Final Shape: (59184, 17) ---
--- Generating v4 set: Train=2019-01-01to2021-12-31, Val=2022-01-01to2022-12-31 ---
--- Block for 2022-01-01 complete. Final Shape: (15695, 17) ---

--- All 2 Folds Created. Aligning columns... ---

--- Cross-Validation Folds Ready ---
Total features (after one-hot): 11
Fold 1 (Val 2023): Train=63568, Val=20075
Fold 2 (Val 2022): Train=59184, Val=15695


## Step 3: Optuna Tuning with Final CV (XGBoost)

Now we will run our Optuna study on these 2, full-year, robust folds. We will run **50 trials** (50 * 2 folds = 100 models). This will give us our most trustworthy hyperparameters.

In [24]:
print(f"--- Starting XGBoost CV Tuning (Final Folds) ---")

def objective_xgb_cv_final(trial):
    params = {
        'objective': 'reg:quantileerror',
        'quantile_alpha': 0.2,
        'n_jobs': -1,
        'seed': 42,
        'eval_metric': 'rmse',
        
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10)
    }
    
    fold_scores = []
    
    for i, (X_t, y_t, X_v, y_v) in enumerate(cv_folds):
        
        dtrain_cv = xgb.DMatrix(X_t, label=y_t)
        dval_cv = xgb.DMatrix(X_v, label=y_v)
        
        model_cv = xgb.train(
            params,
            dtrain_cv,
            num_boost_round=1000,
            evals=[(dval_cv, 'validation')],
            custom_metric=xgb_native_quantile_error,
            callbacks=[xgb.callback.EarlyStopping(rounds=50,
                                                  metric_name='q0.2_error',
                                                  maximize=False,
                                                  save_best=True)],
            verbose_eval=False
        )
        
        y_pred_cv = model_cv.predict(dval_cv)
        score = quantile_error_0_2_raw(y_v, y_pred_cv)
        fold_scores.append(score)
    
    avg_score = np.mean(fold_scores)
    return avg_score

# --- Run the CV Tuning Study ---
print("\n--- 2. Running Optuna study with 2-Fold 'Full-Year' CV... ---")
study_xgb_cv = optuna.create_study(direction='minimize')
study_xgb_cv.optimize(objective_xgb_cv_final, n_trials=50) # 50 trials

# --- 3. Report Best Results ---
print("\n--- Optuna XGB CV (Final) Study Complete ---")
print(f"Number of finished trials: {len(study_xgb_cv.trials)}")
print("Best trial (based on 2-fold Full-Year CV average):")
best_trial_xgb_cv = study_xgb_cv.best_trial

print(f"  Value (Best CV Score): {best_trial_xgb_cv.value:.2f}")
print("  Params: ")
for key, value in best_trial_xgb_cv.params.items():
    print(f"    {key}: {value}")

print("\nThis new CV score is our *most reliable* recipe.")

[I 2025-11-07 02:31:37,308] A new study created in memory with name: no-name-a05407d1-7d8e-421c-8aeb-3005bb1ef1bb


--- Starting XGBoost CV Tuning (Final Folds) ---

--- 2. Running Optuna study with 2-Fold 'Full-Year' CV... ---


[I 2025-11-07 02:31:37,976] Trial 0 finished with value: 153194.51666962315 and parameters: {'learning_rate': 0.016849710772528857, 'max_depth': 7, 'subsample': 0.7891975493937271, 'colsample_bytree': 0.7767088937605103, 'min_child_weight': 2}. Best is trial 0 with value: 153194.51666962315.
[I 2025-11-07 02:31:38,399] Trial 1 finished with value: 151397.04039797088 and parameters: {'learning_rate': 0.0516558451704263, 'max_depth': 5, 'subsample': 0.7167593711557274, 'colsample_bytree': 0.9697510383906589, 'min_child_weight': 8}. Best is trial 1 with value: 151397.04039797088.
[I 2025-11-07 02:31:39,225] Trial 2 finished with value: 151376.19039502146 and parameters: {'learning_rate': 0.01144952370634152, 'max_depth': 8, 'subsample': 0.9868647531821109, 'colsample_bytree': 0.7697198000717431, 'min_child_weight': 2}. Best is trial 2 with value: 151376.19039502146.
[I 2025-11-07 02:31:40,168] Trial 3 finished with value: 159472.79081839527 and parameters: {'learning_rate': 0.010856826119


--- Optuna XGB CV (Final) Study Complete ---
Number of finished trials: 50
Best trial (based on 2-fold Full-Year CV average):
  Value (Best CV Score): 147850.19
  Params: 
    learning_rate: 0.04622027732226668
    max_depth: 8
    subsample: 0.8679610287828807
    colsample_bytree: 0.9909044096241187
    min_child_weight: 8

This new CV score is our *most reliable* recipe.


## Step 4: Train Final Model (Stage 2)

This is **Stage 2** of our plan. We now have the best "recipe" from our robust CV.

We will now:
1.  Create one final, massive training set using our most recent data: **2021, 2022, 2023, and 2024**.
2.  Train our final XGBoost model on this *entire* set, using the best hyperparameters we just found.
3.  Generate `xgboost_v3_final.csv` from this model.

In [26]:
print(f"--- Starting Stage 2: Training Final Model ---")

# --- 1. Get Best "Recipe" ---
try:
    best_params = best_trial_xgb_cv.params.copy()
    print(f"Loaded best parameters from CV: {best_params}")
    
    # Add fixed params
    best_params['objective'] = 'reg:quantileerror'
    best_params['quantile_alpha'] = 0.2
    best_params['n_jobs'] = -1
    best_params['seed'] = 42
    
except NameError:
    print("ERROR: 'best_trial_xgb_cv' not found. Did Cell 4 run correctly?")
    raise
    
# --- 2. Create Final Training Set (2021-2024) ---
print("\n--- 2. Building final 2021-2024 training set... ---")
# Per our plan, we will train on 2021, 2022, 2023, and 2024 data.
# This requires building 4 blocks of data and stacking them.

# Block 1: Train 2018-2020, Targets 2021
block_2021_f, block_2021_u = create_feature_set_v4('2018-01-01', '2020-12-31', '2021-01-01', '2021-12-31')
# Block 2: Train 2019-2021, Targets 2022
block_2022_f, block_2022_u = create_feature_set_v4('2019-01-01', '2021-12-31', '2022-01-01', '2022-12-31')
# Block 3: Train 2020-2022, Targets 2023
block_2023_f, block_2023_u = create_feature_set_v4('2020-01-01', '2022-12-31', '2023-01-01', '2023-12-31')
# Block 4: Train 2021-2023, Targets 2024
block_2024_f, block_2024_u = create_feature_set_v4('2021-01-01', '2023-12-31', '2024-01-01', '2024-12-31')

# Stack the *filtered* dataframes to create our final training set
df_train_final = pd.concat([block_2021_f, block_2022_f, block_2023_f, block_2024_f], ignore_index=True)
print(f"Final training set created. Shape: {df_train_final.shape}")

# --- 3. One-hot encode our final training data ---
print("One-hot encoding final training data...")
df_train_final_encoded = pd.get_dummies(df_train_final, columns=CATEGORICAL_COLS, prefix_sep='_f_')

# Align with our master list of features from the CV
df_train_final_aligned, _ = df_train_final_encoded.align(master_cols_df, join='right', axis=1, fill_value=0)

X_full_final = df_train_final_aligned[ALL_FEATURES]
y_full_final = df_train_final_aligned[TARGET_COL]

# Create the full DMatrix for final training
dfull_final = xgb.DMatrix(X_full_final, label=y_full_final)
print(f"Final DMatrix created. Shape: {X_full_final.shape}")

# --- 4. Find Optimal n_estimators ---
print("\n--- 4. Finding optimal n_estimators for final data... ---")
best_params.pop('n_estimators', None) 

cv_results = xgb.cv(
    params=best_params,
    dtrain=dfull_final,
    num_boost_round=1000,
    nfold=3, # 3-fold CV on our final dataset
    metrics={'rmse'}, 
    early_stopping_rounds=50,
    seed=42,
    verbose_eval=False
)

BEST_FINAL_ITERATION = cv_results['test-rmse-mean'].idxmin() + 1
print(f"Found best final iteration: {BEST_FINAL_ITERATION}")

# --- 5. Train Final Model ---
best_params['n_estimators'] = BEST_FINAL_ITERATION
model_final = xgb.train(
    best_params,
    dfull_final,
    num_boost_round=BEST_FINAL_ITERATION
)
print("Final model trained.")

# --- 6. Load and Build Test Set ---
print("\n--- 6. Building Test Set for Submission ---")
# For the test set, we build features from ALL available history (2004-2024)
# to give the model the most info possible for its 2025 prediction.
hist_receivals_test = df_receivals_cleaned.copy() 
hist_po_base_test = df_po_cleaned.copy()
hist_po_test = pd.merge(
    hist_po_base_test,
    df_transport_cleaned[['rm_id', 'purchase_order_id', 'purchase_order_item_no', 'transporter_name']],
    on=['rm_id', 'purchase_order_id', 'purchase_order_item_no'],
    how='left'
)
hist_po_test['transporter_name'] = hist_po_test['transporter_name'].fillna('Transporter_Unknown')

df_test = pd.read_csv("data/prediction_mapping.csv")
df_test['forecast_end_date'] = pd.to_datetime(df_test['forecast_end_date'], utc=True)
TEST_START_DATE = pd.to_datetime('2025-01-01', utc=True)

# Build features
hist_po_test['f_po_lead_time_days'] = (hist_po_test['delivery_date'] - hist_po_test['created_date_time']).dt.days
po_agg = hist_po_test.groupby(['rm_id', 'delivery_date']).agg(daily_po_quantity=('quantity', 'sum'), avg_lead_time=('f_po_lead_time_days', 'mean')).reset_index()
po_agg.rename(columns={'delivery_date': 'po_delivery_date'}, inplace=True)
rm_ids_to_merge = df_test['rm_id'].unique()
merged_df = pd.merge(df_test[['rm_id', 'forecast_end_date']], po_agg[po_agg['rm_id'].isin(rm_ids_to_merge)], on='rm_id', how='left')
merged_df_filtered = merged_df[merged_df['forecast_end_date'] >= merged_df['po_delivery_date']].copy()
cumulative_features = merged_df_filtered.groupby(['rm_id', 'forecast_end_date']).agg(f_cumulative_po_quantity=('daily_po_quantity', 'sum'), f_avg_lead_time=('avg_lead_time', 'mean')).reset_index()
df_test = pd.merge(df_test, cumulative_features, on=['rm_id', 'forecast_end_date'], how='left')
df_test['f_cumulative_po_quantity'] = df_test['f_cumulative_po_quantity'].fillna(0)
df_test['f_avg_lead_time'] = df_test['f_avg_lead_time'].fillna(0)
df_test['f_month'] = df_test['forecast_end_date'].dt.month
df_test['f_day_of_week'] = df_test['forecast_end_date'].dt.dayofweek
df_test['f_day_of_year'] = df_test['forecast_end_date'].dt.dayofyear
df_test['f_is_month_end'] = df_test['forecast_end_date'].dt.is_month_end.astype(str)
df_train_merged_eda = pd.merge(hist_receivals_test, hist_po_test[['purchase_order_id', 'purchase_order_item_no', 'delivery_date', 'rm_id']], on=['purchase_order_id', 'purchase_order_item_no', 'rm_id'], how='inner')
df_train_merged_eda['delivery_lag_days'] = (df_train_merged_eda['date_arrival'] - df_train_merged_eda['delivery_date']).dt.days
rm_id_lag_map = df_train_merged_eda.groupby('rm_id')['delivery_lag_days'].median().reset_index(name='f_median_lag_days')
df_test = pd.merge(df_test, rm_id_lag_map, on='rm_id', how='left')
df_test['f_median_lag_days'] = df_test['f_median_lag_days'].fillna(0)
for days in [30, 90, 180]:
    hist_start_date_window = TEST_START_DATE - pd.Timedelta(days=days)
    window_data = hist_receivals_test[(hist_receivals_test['date_arrival'] >= hist_start_date_window) & (hist_receivals_test['date_arrival'] < TEST_START_DATE)]
    feature_name = f'f_receivals_{days}d'
    feature_map = window_data.groupby('rm_id')['net_weight'].sum().reset_index(name=feature_name)
    df_test = pd.merge(df_test, feature_map, on='rm_id', how='left')
    df_test[feature_name] = df_test[feature_name].fillna(0)
rm_id_static_features = hist_po_test.drop_duplicates(subset=['rm_id'], keep='last')[['rm_id', 'status', 'raw_material_format_type', 'transporter_name']]
df_test = pd.merge(df_test, rm_id_static_features, on='rm_id', how='left')
df_test['status'] = df_test['status'].fillna('Unknown')
df_test['raw_material_format_type'] = df_test['raw_material_format_type'].fillna('Unknown')
df_test['transgitor_name'] = df_test.get('transporter_name', pd.Series(index=df_test.index, name='transporter_name')).fillna('Unknown')

# One-hot encode and align
print("Aligning test features...")
df_test = pd.get_dummies(df_test, columns=CATEGORICAL_COLS, prefix_sep='_f_')
X_test_aligned, _ = df_test.align(master_cols_df, join='right', axis=1, fill_value=0)
X_test = X_test_aligned[ALL_FEATURES]
dtest = xgb.DMatrix(X_test)

# --- 7. Predict & Save ---
print("\n--- 7. Predicting and Saving final submission... ---")
final_predictions = model_final.predict(dtest)
final_predictions[final_predictions < 0] = 0

df_submission = pd.DataFrame({
    'ID': df_test['ID'],
    'predicted_weight': final_predictions
})

SUBMISSION_FILE = "xgboost_v_final_CV.csv"
df_submission.to_csv(SUBMISSION_FILE, index=False)
print(f"--- Final submission '{SUBMISSION_FILE}' created successfully! ---")

--- Starting Stage 2: Training Final Model ---
Loaded best parameters from CV: {'learning_rate': 0.04622027732226668, 'max_depth': 8, 'subsample': 0.8679610287828807, 'colsample_bytree': 0.9909044096241187, 'min_child_weight': 8}

--- 2. Building final 2021-2024 training set... ---
--- Generating v4 set: Train=2018-01-01to2020-12-31, Val=2021-01-01to2021-12-31 ---
--- Block for 2021-01-01 complete. Final Shape: (13870, 17) ---
--- Generating v4 set: Train=2019-01-01to2021-12-31, Val=2022-01-01to2022-12-31 ---
--- Block for 2022-01-01 complete. Final Shape: (15695, 17) ---
--- Generating v4 set: Train=2020-01-01to2022-12-31, Val=2023-01-01to2023-12-31 ---
--- Block for 2023-01-01 complete. Final Shape: (20075, 17) ---
--- Generating v4 set: Train=2021-01-01to2023-12-31, Val=2024-01-01to2024-12-31 ---
--- Block for 2024-01-01 complete. Final Shape: (21960, 17) ---
Final training set created. Shape: (71600, 17)
One-hot encoding final training data...
Final DMatrix created. Shape: (71600, 

/Users/jennarx/Desktop/NTNU/ModernMachineLearning/group_project/.venv/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [02:34:12] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "n_estimators" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Final model trained.

--- 6. Building Test Set for Submission ---
Aligning test features...

--- 7. Predicting and Saving final submission... ---
--- Final submission 'xgboost_v_final_CV.csv' created successfully! ---
